# Y = ReLU(XW + b): PyTorch 2 → Triton → PTX → SASS on GPU

`torch.compile`이 식 하나를 **Dynamo → AOT Autograd → Inductor → Triton 컴파일러 → PTX/SASS**로 내리는 과정을 단계별로 출력한다.

**런타임 설정**: 메뉴 `런타임 → 런타임 유형 변경 → A100 GPU` (기준 문서는 A100). T4 등 다른 GPU에서도 동작하지만 PTX의 `sm_XX`, 캐시 힌트 명령, SASS 명령 수, 5절의 속도가 달라진다.

문서: 저장소의 `docs/example-pytorch-gpu-a100.md` 의 각 절 번호와 셀 제목이 대응한다.

## 0. 환경 확인과 로그 설정

Inductor 캐시 위치를 고정해 뒤에서 Triton 중간 산출물을 찾을 수 있게 한다. **`torch`를 import하기 전에** 환경변수를 넣어야 한다.

In [ ]:
import os
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/tmp/inductor_cache_demo"
os.environ["TORCHINDUCTOR_FORCE_DISABLE_CACHES"] = "1"   # 매번 새로 컴파일해서 로그가 항상 찍히게

import torch, triton
assert torch.cuda.is_available(), "런타임 유형을 GPU로 바꾸세요"
p = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| triton", triton.__version__)
print(p.name, "| SMs", p.multi_processor_count, "| HBM", round(p.total_memory / 2**30, 1), "GiB | cc", f"{p.major}.{p.minor}")

In [ ]:
# 단계별 로그를 켠다. 로그는 stderr로 나오므로 Colab에서는 셀 출력에 붉은 배경으로 표시된다.
torch._logging.set_logs(graph_code=True, guards=True, aot_graphs=True, output_code=True)

## 1~3. 컴파일: Dynamo FX 그래프, guard, AOT forward/backward, Inductor 출력 코드

아래 셀 하나가 실행되면 로그에 순서대로 나온다.

- `TRACED GRAPH` (Dynamo FX 그래프, `cuda:0` 타입 주석)
- `GUARDS:` 트리 (`TENSOR_MATCH ... device=0`, `ID_MATCH torch.relu`)
- `Forward graph 0` / `Backward graph 0` (ATen 수준, `le`·`permute`가 backward용으로 저장됨)
- `Output code:` forward — `extern_kernels.mm` + `triton_poi_fused_add_relu_threshold_backward_0`
- `Output code:` backward — `triton_per_fused_sum_threshold_backward_0` + `extern_kernels.mm`

In [ ]:
def f(x, w, b):
    return torch.relu(x @ w + b)

x = torch.ones(16, 8, device="cuda")
w = torch.ones(8, 4, device="cuda", requires_grad=True)
b = torch.ones(4, device="cuda", requires_grad=True)

cf = torch.compile(f)
y = cf(x, w, b)
y.sum().backward()

print("=== RESULT ===")
print(type(y), y.shape, y.device)
print("grad_fn:", y.grad_fn)
print("w.grad.shape:", w.grad.shape, "| b.grad:", b.grad)

## 4. Triton 컴파일러 체인: TTIR → TTGIR → LLVM IR → PTX → SASS

Inductor 캐시 디렉터리에 forward 커널의 모든 중간 산출물이 남는다.

In [ ]:
import glob, subprocess
torch._logging.set_logs()   # 이후 셀에서는 로그 끔

name = "triton_poi_fused_add_relu_threshold_backward_0"
hits = glob.glob(f"{os.environ['TORCHINDUCTOR_CACHE_DIR']}/**/{name}.ptx", recursive=True)
assert hits, "캐시에서 커널을 못 찾았습니다. 위 셀을 먼저 실행하세요."
d = os.path.dirname(hits[0])
print("cache dir:", d)
print(sorted(os.path.basename(p) for p in glob.glob(d + "/*")))

In [ ]:
# 4-1. TTIR: 하드웨어 독립 Triton IR. 값이 tensor<64xf32> — 스레드 개념이 없다.
print(open(f"{d}/{name}.ttir").read())

In [ ]:
# 4-2. TTGIR: #blocked 레이아웃(sizePerThread, threadsPerWarp, warpsPerCTA)과 ttg.target이 붙는다.
print(open(f"{d}/{name}.ttgir").read())

In [ ]:
# 4-3. PTX: %ctaid.x = program_id, %tid.x = 스레드, @%p 는 마스크 predicate, ld.global.v2 = 스레드당 2원소
print(open(f"{d}/{name}.ptx").read())

In [ ]:
# 4-4. SASS: 실제 GPU 명령. S2R SR_TID.X / SR_CTAID.X, LDG, FADD, FSEL, STG
r = subprocess.run(["cuobjdump", "-sass", f"{d}/{name}.cubin"], capture_output=True, text=True)
sass = [l for l in r.stdout.splitlines() if l.strip().startswith("/*0")]
print(f"{len(sass)} instructions")
print(r.stdout)

## 5. 비동기 디스패치

Python 함수가 돌아오는 시간과 커널이 끝나는 시간은 다르다.

In [ ]:
import time
a = torch.randn(4096, 4096, device="cuda")
for _ in range(3): a @ a            # 웜업 (cuBLAS 초기화 포함)
torch.cuda.synchronize()

t0 = time.perf_counter(); z = a @ a; t1 = time.perf_counter()
torch.cuda.synchronize();          t2 = time.perf_counter()
print(f"dispatch only          : {(t1 - t0) * 1e6:8.1f} us")
print(f"until synchronize()    : {(t2 - t0) * 1e3:8.3f} ms")

t0 = time.perf_counter(); z = a @ a; z.sum().item(); t1 = time.perf_counter()
print(f".item() (implicit sync): {(t1 - t0) * 1e3:8.3f} ms")

# Tensor Core 비교: 같은 x @ x 를 어느 유닛이 계산하는가 (A100 이상에서 TF32 사용 가능)
flops = 2 * 4096**3
def bench(fn):
    for _ in range(3): fn()
    torch.cuda.synchronize(); t0 = time.perf_counter(); fn(); torch.cuda.synchronize()
    return time.perf_counter() - t0
torch.backends.cuda.matmul.allow_tf32 = False
s = bench(lambda: a @ a); print(f"f32  allow_tf32=False : {s*1e3:7.3f} ms  {flops/s/1e12:6.1f} TFLOP/s  (CUDA core)")
torch.backends.cuda.matmul.allow_tf32 = True
s = bench(lambda: a @ a); print(f"f32  allow_tf32=True  : {s*1e3:7.3f} ms  {flops/s/1e12:6.1f} TFLOP/s  (TF32 Tensor Core, Ampere+)")
torch.backends.cuda.matmul.allow_tf32 = False
ab = a.to(torch.bfloat16)
s = bench(lambda: ab @ ab); print(f"bf16                  : {s*1e3:7.3f} ms  {flops/s/1e12:6.1f} TFLOP/s  (bf16 Tensor Core)")


## 6. 직접 써 보는 Triton 커널

Inductor가 만든 것과 같은 구조를 손으로 쓰면 이렇다. `program_id`, `arange`, 마스크, `load`/`store`.

In [ ]:
import triton.language as tl

@triton.jit
def relu_bias_kernel(x_ptr, b_ptr, out_ptr, n_elements, n_cols, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    xv = tl.load(x_ptr + offs, mask=mask)
    bv = tl.load(b_ptr + (offs % n_cols), mask=mask)
    tl.store(out_ptr + offs, tl.maximum(xv + bv, 0.0), mask=mask)

xw = (x @ w).detach()
out = torch.empty_like(xw)
grid = lambda meta: (triton.cdiv(xw.numel(), meta["BLOCK"]),)
relu_bias_kernel[grid](xw, b.detach(), out, xw.numel(), xw.shape[1], BLOCK=64)
print("matches torch.compile result:", torch.allclose(out, y.detach()))